Co-training is a semi-supervised machine learning technique where two (or more) classifiers are trained simultaneously on the same dataset but using different views (feature sets) of the data. The idea is to leverage a small amount of labeled data and a large amount of unlabeled data to improve learning.

Key points about Co-training:
Two classifiers are trained on two different and ideally complementary feature sets (called views) of the same data.

Each classifier labels some unlabeled data that it is confident about.

These newly labeled samples are then added to the training set of the other classifier.

The process repeats iteratively, improving both classifiers by exchanging information.

In [13]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score

# Generate synthetic data
np.random.seed(42)
X = np.random.rand(200, 6)  # 200 samples, 6 features
y = np.random.choice([0, 1], size=200)  # Binary labels

# Split data into labeled and unlabeled sets
X_train, X_unlabeled, y_train, y_unlabeled = train_test_split(X, y, test_size=0.7, stratify=y, random_state=42)
y_unlabeled[:] = -1  # Mask unlabeled data

# Split features for Co-Training
X_train_1, X_train_2 = X_train[:, :3], X_train[:, 3:]
X_unlabeled_1, X_unlabeled_2 = X_unlabeled[:, :3], X_unlabeled[:, 3:]

# Initialize classifiers
clf1 = GaussianNB()
clf2 = GaussianNB()

# Train on initial labeled data
clf1.fit(X_train_1, y_train)
clf2.fit(X_train_2, y_train)

# Co-Training process
max_iterations = 10
confidence_threshold = 0.5

for _ in range(max_iterations):
    # Check if there are any unlabeled data points remaining
    if len(X_unlabeled_1) == 0 or len(X_unlabeled_2) == 0:
        print("No unlabeled data remaining. Stopping Co-Training.")
        break

    # Get predictions and confidence scores
    prob_1 = clf1.predict_proba(X_unlabeled_1)
    prob_2 = clf2.predict_proba(X_unlabeled_2)
    
    high_conf_1 = np.max(prob_1, axis=1) > confidence_threshold
    high_conf_2 = np.max(prob_2, axis=1) > confidence_threshold

    # Label new samples with high confidence
    new_labels_1 = clf1.predict(X_unlabeled_1[high_conf_1])
    new_labels_2 = clf2.predict(X_unlabeled_2[high_conf_2])

    # Stop if no high-confidence labels
    if len(new_labels_1) == 0 and len(new_labels_2) == 0:
        print("No high-confidence labels found. Stopping Co-Training.")
        break

    # Add new labeled data to training set
    if len(new_labels_1) > 0:
        X_train_1 = np.vstack([X_train_1, X_unlabeled_1[high_conf_1]])
        y_train_1 = np.concatenate([y_train, new_labels_1])
        X_unlabeled_1 = np.delete(X_unlabeled_1, np.where(high_conf_1), axis=0)

    if len(new_labels_2) > 0:
        X_train_2 = np.vstack([X_train_2, X_unlabeled_2[high_conf_2]])
        y_train_2 = np.concatenate([y_train, new_labels_2])
        X_unlabeled_2 = np.delete(X_unlabeled_2, np.where(high_conf_2), axis=0)

    # Retrain classifiers only if there are new samples
    if len(new_labels_1) > 0:
        clf1.fit(X_train_1, y_train_1)

    if len(new_labels_2) > 0:
        clf2.fit(X_train_2, y_train_2)

# Evaluate performance on test set
X_test, X_test_labels, y_test, _ = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
X_test_1, X_test_2 = X_test[:, :3], X_test[:, 3:]

y_pred_1 = clf1.predict(X_test_1)
y_pred_2 = clf2.predict(X_test_2)

final_pred = (y_pred_1 + y_pred_2) // 2  # Majority voting

print("Co-Training Accuracy:", accuracy_score(y_test, final_pred))


No unlabeled data remaining. Stopping Co-Training.
Co-Training Accuracy: 0.5375


In [19]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score

# Generate synthetic data with structure
np.random.seed(42)

# Create a feature matrix X with 200 samples and 6 features
X = np.random.rand(200, 6)

# Create a structured class label 'y' based on a linear combination of features
# Class 0 if sum of first 3 features is less than 1.5, else class 1
y = (X[:, 0] + X[:, 1] + X[:, 2]) < 1.5  # This creates a more structured decision boundary

# Convert boolean labels to integers (0 or 1)
y = y.astype(int)

# Split data into labeled and unlabeled sets
X_train, X_unlabeled, y_train, y_unlabeled = train_test_split(X, y, test_size=0.7, stratify=y, random_state=42)
y_unlabeled[:] = -1  # Mask unlabeled data

# Split features for Co-Training
X_train_1, X_train_2 = X_train[:, :3], X_train[:, 3:]
X_unlabeled_1, X_unlabeled_2 = X_unlabeled[:, :3], X_unlabeled[:, 3:]

# Initialize classifiers
clf1 = GaussianNB()
clf2 = GaussianNB()

# Train on initial labeled data
clf1.fit(X_train_1, y_train)
clf2.fit(X_train_2, y_train)

# Co-Training process
max_iterations = 10
confidence_threshold = 0.5

for _ in range(max_iterations):
    # Check if there are any unlabeled data points remaining
    if len(X_unlabeled_1) == 0 or len(X_unlabeled_2) == 0:
        print("No unlabeled data remaining. Stopping Co-Training.")
        break

    # Get predictions and confidence scores
    prob_1 = clf1.predict_proba(X_unlabeled_1)
    prob_2 = clf2.predict_proba(X_unlabeled_2)
    
    high_conf_1 = np.max(prob_1, axis=1) > confidence_threshold
    high_conf_2 = np.max(prob_2, axis=1) > confidence_threshold

    # Label new samples with high confidence
    new_labels_1 = clf1.predict(X_unlabeled_1[high_conf_1])
    new_labels_2 = clf2.predict(X_unlabeled_2[high_conf_2])

    # Stop if no high-confidence labels
    if len(new_labels_1) == 0 and len(new_labels_2) == 0:
        print("No high-confidence labels found. Stopping Co-Training.")
        break

    # Add new labeled data to training set
    if len(new_labels_1) > 0:
        X_train_1 = np.vstack([X_train_1, X_unlabeled_1[high_conf_1]])
        y_train_1 = np.concatenate([y_train, new_labels_1])
        X_unlabeled_1 = np.delete(X_unlabeled_1, np.where(high_conf_1), axis=0)

    if len(new_labels_2) > 0:
        X_train_2 = np.vstack([X_train_2, X_unlabeled_2[high_conf_2]])
        y_train_2 = np.concatenate([y_train, new_labels_2])
        X_unlabeled_2 = np.delete(X_unlabeled_2, np.where(high_conf_2), axis=0)

    # Retrain classifiers only if there are new samples
    if len(new_labels_1) > 0:
        clf1.fit(X_train_1, y_train_1)

    if len(new_labels_2) > 0:
        clf2.fit(X_train_2, y_train_2)

# Evaluate performance on test set
X_test, X_test_labels, y_test, _ = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
X_test_1, X_test_2 = X_test[:, :3], X_test[:, 3:]

y_pred_1 = clf1.predict(X_test_1)
y_pred_2 = clf2.predict(X_test_2)

final_pred = (y_pred_1 + y_pred_2) // 2  # Majority voting

print("Co-Training Accuracy:", accuracy_score(y_test, final_pred))


No unlabeled data remaining. Stopping Co-Training.
Co-Training Accuracy: 0.78125
